In [ ]:
from utils import paths
import pypsa
import numpy as np
import pandas as pd
from utils.plotting import draw_network
import matplotlib.pyplot as plt
import geopandas as gpd
shapes = gpd.read_file(paths.COUNTRY_SHAPES)

# Description du réseau électrique européen

## Réseau sans générateurs

### Réseau complet

In [ ]:
n = pypsa.Network(paths.BASE_NETWORK)

fig, ax = plt.subplots(figsize=(15, 10))
draw_network(ax, n, shapes)
ax.set_title("Réseau haute tension de PyPSA-Eur")

### Réseau simplifié et ne contenant que la zone synchrone continentale

In [ ]:
from utils.network import remove_non_synchronous_areas

n = pypsa.Network(paths.CLUSTERED_NETWORK)

n = remove_non_synchronous_areas(n)
fig, ax = plt.subplots(figsize=(15, 10))
draw_network(ax, n, shapes)
ax.set_title("Réseau haute tension simplifié")

In [ ]:
print("Nombre de noeuds : ", len(n.buses))
print("Nombre de lignes : ", len(n.lines))

## Ajout des générateurs aux noeuds du réseau

In [ ]:
from utils.grid import Grid

# Réseau synchrone continental + inerties des centrales
grid = Grid(paths.CLUSTERED_NETWORK, paths.POWERPLANTS)

In [ ]:
print(f"Nombre de noeuds générateurs : {len(grid.buses[grid.buses['S'] > 0].index)} / {len(grid.buses.index)}")

In [ ]:
grid.buses[['country', 'S']].groupby('country').sum().sort_values('S', ascending=False).apply(lambda x: x/1000).iloc[:10].plot(kind='bar', figsize=(10, 5), legend=False)
plt.title("Capacité de production par pays (top 10)")
plt.ylabel("Capacité de production (GW)")
plt.xlabel("Pays")
plt.show()

In [ ]:
from utils.plotting import draw_graph_signals_area

fig, ax = plt.subplots(figsize=(12, 8))
draw_graph_signals_area(ax, grid, grid.buses.E, shapes)
ax.set_title("Inertie des noeuds du réseau (proportionnelle à la surface des nœuds)")
plt.show()

# Analyse modale du réseau

In [ ]:
from matplotlib.colors import CenteredNorm
from utils.plotting import draw_graph_signals

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for k, ax in enumerate(axes.flat, start=1):
    sc = draw_graph_signals(ax, grid, grid.modes[:, k] / np.abs(grid.modes[:, k]).max(), shapes,
                            norm=CenteredNorm(halfrange=1))
    ax.set_title(f"Mode {k} : {grid.frequencies[k]:.3f} Hz")
fig.colorbar(sc, ax=axes, shrink=0.6, label="Forme du mode (normalisée)")

In [ ]:
from utils.plotting import plot_mode_3d
fig = plot_mode_3d(grid, grid.modes[:, 1], shapes, mode_index=1)
fig.show()

In [ ]:
from matplotlib.colors import CenteredNorm
import utils.simulation as sim

# Perte de production de 10 MW sur chaque nœud à l'ouest de -8° (Portugal, Galice)
Delta_P = np.where(grid.buses.x < -8, -10.0, 0.0)
print(f"ΔP total : {Delta_P.sum():.0f} MW")

# Une période du premier mode inter-zones (~0.22 Hz -> ~4.5 s)
t = np.linspace(0, 4.5, 10)
delta, freq = sim.step_response(grid, Delta_P, t)

# Écart à la fréquence du centre d'inertie : on retire la dérive commune pour voir les oscillations entre zones
m = grid.M.diagonal()
f_coi = m @ freq.to_numpy() / m.sum()
freq_rel = freq - f_coi

# Même échelle de couleurs pour tous les instants
norm = CenteredNorm(halfrange=np.abs(freq_rel.to_numpy()).max() * 1e3)
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
for i, ax in enumerate(axes.flat):
    draw_graph_signals(ax, grid, freq_rel.iloc[:, i] * 1e3, shapes, norm=norm)
    ax.set_title(f"t = {t[i]:.1f} s")
fig.suptitle("Écart de fréquence au centre d'inertie (mHz)")

In [ ]:
grid.set_rayleigh_coefficients(alpha=0, beta=0.001)

In [ ]:
# Évolution temporelle : centre d'inertie et nœuds aux deux extrémités du réseau
t_fine = np.linspace(0, 10, 1001)
_, freq_fine = sim.step_response(grid, Delta_P, t_fine)
f_coi_fine = m @ freq_fine.to_numpy() / m.sum()
portugal_avg = freq_fine.loc[grid.buses.x < -8].mean(axis=0)
west, east = grid.buses.x.idxmin(), grid.buses.x.idxmax()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_fine, freq_fine.loc[west] * 1e3, label=f"Ouest ({grid.buses.country[west]})")
ax.plot(t_fine, freq_fine.loc[east] * 1e3, label=f"Est ({grid.buses.country[east]})")
ax.plot(t_fine, portugal_avg * 1e3, label=f"Portugal (moyenne)")
ax.plot(t_fine, f_coi_fine * 1e3, "k--", label="Centre d'inertie")
ax.set_xlabel("t (s)")
ax.set_ylabel("Δf (mHz)")
ax.legend()
ax.set_title(f"RoCoF du centre d'inertie : {f_coi_fine[-1] / t_fine[-1] * 1e3:.1f} mHz/s")

In [ ]:
# Animation des oscillations entre zones, enregistrée en GIF (pas intégrée au notebook pour limiter sa taille)
from utils.plotting import animate_graph_signals

t_anim = np.linspace(0, 10, 201)
_, freq_anim = sim.step_response(grid, Delta_P, t_anim)
f_coi_anim = m @ freq_anim.to_numpy() / m.sum()

anim = animate_graph_signals(grid, (freq_anim - f_coi_anim) * 1e3, shapes,
                             label="Écart au centre d'inertie (mHz)")
anim.save("oscillations.gif", writer="pillow", fps=20, dpi=100)